# Ames Housing - Neural Network regressie

Doel: met een neural network de huizenprijzen (`SalePrice`) voorspellen en onderzoeken welke feature set het beste werkt. De resultaten worden vergeleken met lineaire regressie op dezelfde vaste testset.

De experimenten staan in `run_ames_nn_experiments.py`. Dat script draait 540 NN-experimenten met verschillende feature sets, validation-groottes, learning rates, epochs en hidden-layer configuraties.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

BASE_DIR = Path.cwd()
if not (BASE_DIR / "run_ames_nn_experiments.py").exists():
    BASE_DIR = Path.cwd() / "Week 9 ML"

RESULTS_DIR = BASE_DIR / "ames_nn_results"
RESULTS_DIR

## Experimenten draaien

Deze cel hoeft alleen opnieuw uitgevoerd te worden als je de experimenten opnieuw wilt berekenen. De huidige resultaten zijn al opgeslagen in `ames_nn_results/`.

In [ ]:
# Duurt enkele minuten: 540 experimenten.
# %run run_ames_nn_experiments.py

## Resultaten laden

In [ ]:
nn_results = pd.read_csv(RESULTS_DIR / "ames_nn_experiment_results.csv")
comparison = pd.read_csv(RESULTS_DIR / "ames_nn_vs_linear_comparison.csv")

print("Aantal NN-experimenten:", len(nn_results))
print("Feature sets:", nn_results["feature_set"].nunique())
nn_results.head()

## Beste neural-network modellen

Modelkeuze gebeurt op basis van de validation RMSE. Daarna bekijken we de test RMSE en R2 als eindcontrole.

In [ ]:
top_10 = nn_results.sort_values("validation_rmse").head(10)
top_10[[
    "feature_set",
    "validation_rmse",
    "validation_mae",
    "validation_r2",
    "test_rmse",
    "test_mae",
    "test_r2",
    "architecture",
    "learning_rate",
    "epochs",
    "validation_size",
]]

In [ ]:
img = plt.imread(RESULTS_DIR / "top_20_nn_validation_rmse.png")
plt.figure(figsize=(12, 8))
plt.imshow(img)
plt.axis("off");

## Feature sets vergelijken

In [ ]:
comparison_sorted = comparison.sort_values("validation_rmse")
comparison_sorted[[
    "feature_set",
    "validation_rmse",
    "test_rmse_nn",
    "test_rmse_linear",
    "test_r2_nn",
    "test_r2_linear",
    "architecture",
    "learning_rate",
    "epochs",
    "validation_size",
    "rmse_verschil_nn_min_linear",
]]

In [ ]:
img = plt.imread(RESULTS_DIR / "best_feature_sets_validation_rmse.png")
plt.figure(figsize=(10, 5))
plt.imshow(img)
plt.axis("off");

## Conclusie

De beste configuratie op validation RMSE gebruikt `alle_features_zonder_id`: `Garage`, `Overall Qual`, `Gr Liv Area`, `Total Bsmt SF`, `Lot Area`, `Year Built`, `Full Bath`, `Bedroom AbvGr`, `Neighborhood` en `House Style`. De beste setting was een neural network met 3 hidden layers `(128, 64, 32)`, learning rate `0.01`, `300` epochs en validation size `0.15`. Dit model haalde validation RMSE `23230.78`, test RMSE `27923.84` en test R2 `0.9027`.

Alle features leveren dus de beste validation-score op. Toch is het verschil met de `sterke_subset` klein: deze subset gebruikt minder features en haalt validation RMSE `23479.32`, test RMSE `27825.56` en test R2 `0.9034`. Op de testset is die kleinere subset zelfs iets beter, maar omdat de opdracht expliciet een validation set gebruikt voor modelkeuze, is `alle_features_zonder_id` de beste keuze volgens de afgesproken selectieprocedure.

Vergeleken met lineaire regressie presteert het neural network duidelijk beter voor iedere feature set. Voor de beste feature set daalt de test RMSE van `35872.99` bij lineaire regressie naar `27923.84` bij het neural network. De test R2 stijgt daarbij van `0.8395` naar `0.9027`.

De belangrijkste les: meer features helpen hier, maar niet onbeperkt overtuigend. Een compacte subset met kwaliteit, woonoppervlak, kelderoppervlak, bouwjaar, badkamers en buurt komt bijna even ver. Voor uitlegbaarheid zou die subset aantrekkelijk zijn; voor de beste validation-score wint het volledige model zonder `ID`.